In [20]:
using LinearAlgebra
using Random
using Plots

plotly()

Random.seed!(42)

n = 100

A = Float64.(rand(-100:100, n, n)) ./ 100.00
b = Float64.(rand(-500:500, n)) ./ 100.00
delta_A = Float64.(rand(-2:2, n, n)) ./ 100.00
delta_b = Float64.(rand(-10:10, n)) ./ 100.00

# РЕШЕНИЕ СЛАУ
function solveSystem(A, b)
    Acopy = copy(A)
    bcopy = copy(b)
    x, factors, pivots = LAPACK.gesv!(Acopy, bcopy)
    return x
end

# НАХОЖДЕНИЕ ОБРАТНОЙ МАТРИЦЫ
function inverseMatrix(A)
    n = size(A, 1)
    E = Matrix{Float64}(I, n, n)
    Ainv, factors, pivots = LAPACK.gesv!(copy(A), E)
    return Ainv
end

# ЧИСЛО ОБУСЛОВЛЕННОСТИ nu(A)
function condNumber(A, p)
    Ainv = inverseMatrix(A)
    return opnorm(A, p) * opnorm(Ainv, p)
end

# АНАЛИЗ ОШИБКИ
function analyzeError(A, b, delta_A, delta_b, p)
    x = solveSystem(A, b)
    xNew = solveSystem(A + delta_A, b + delta_b)
    delta_x = xNew - x
    nu = condNumber(A, p)
    rel_b = norm(delta_b, p) / norm(b, p)
    rel_A = opnorm(delta_A, p) / opnorm(A, p)

    # Прямой расчет относительной ошибки
    errDirect = norm(delta_x, p) / norm(x, p)
    # Теоретическая оценка
    errEstimate = nu * (rel_b + rel_A)

    return (x, xNew, delta_x, nu, rel_b, rel_A, errDirect, errEstimate)
end

# ВЫВОД РЕЗУЛЬТАТОВ
function printAnalysis(A, b, delta_A, delta_b, p)
    x, xNew, delta_x, nu, rel_b, rel_A, errDirect, errEstimate = analyzeError(A, b, delta_A, delta_b, p)
    
    if p == Inf
        println()
        println("============== p = Inf ==============")
    else
        println()
        println("=============== p = 2 ===============")
    end
    println("Решение системы x")
    println(round.(x, digits = 2))
    println()
    println("Решение системы x~")
    println(round.(xNew, digits = 2))
    println()
    println("delta_x = x~ - x")
    println(round.(delta_x, digits = 2))
    println()
    println("nu(A)")
    println(round(nu, digits = 2))
    println()
    println("||delta_b||p / ||b||p")
    println(round(rel_b, digits = 2))
    println()
    println("||delta_A||p / ||A||p")
    println(round(rel_A, digits = 2))
    println()
    println("Прямая относительная ошибка")
    println(round(errDirect * 100.00, digits = 2)," %")
    println()
    println("Теоретическая оценка")
    println(round(errEstimate * 100.00, digits = 2)," %")
    println()
    println("Прямая ошибка <= Теоретическая оценка: ", errDirect <= errEstimate)
end

# ВЫВОД ИСХОДНЫХ ДАННЫХ
println("Матрица A")
display(round.(A, digits = 2))

println()
println("Вектор b")
println(round.(b, digits = 2))

println()
println("Матрица delta_A")
display(round.(delta_A, digits = 2))

println()
println("Вектор delta_b")
println(round.(delta_b, digits = 2))

# РАСЧЕТ ДЛЯ p = 2
printAnalysis(A, b, delta_A, delta_b, 2)

# РАСЧЕТ ДЛЯ p = Inf
printAnalysis(A, b, delta_A, delta_b, Inf)

Матрица A


100×100 Matrix{Float64}:
  0.26  -0.49   0.52   0.48   0.25  …  -0.03   0.11   0.65  -0.69  -0.36
 -0.1    0.76   0.57   0.93   0.26     -0.92   0.04  -0.25   0.38  -0.94
 -0.05  -0.36   0.41  -0.39   0.05      0.72   0.86   0.41  -0.56   0.51
  0.41   0.25   0.72  -0.02   0.17     -0.55   0.88  -0.96  -0.72   0.67
  0.35   0.29   0.08  -0.56   0.63     -0.95  -0.49   0.71   0.96   0.36
 -0.67  -0.04   0.41   0.93   0.55  …   0.44   0.11  -0.32  -0.66   0.19
  0.23  -0.63   0.48   0.74  -0.08     -0.63   0.26  -0.91  -0.87  -0.94
  0.34   0.6    0.56   0.94   0.42      0.34  -0.59   0.83   0.62   0.33
 -0.09  -0.51  -0.94  -0.57  -0.27     -0.51   0.85  -0.63   0.73  -0.32
 -0.4   -0.92  -0.2    0.71  -0.51      0.79   0.9   -0.9    0.04   0.06
  0.32   0.09   0.72  -0.1    0.5   …  -0.21  -0.58   0.94   0.46  -0.05
  0.28   0.69   0.84   0.96  -0.94     -0.98   0.76  -0.81   0.59   0.49
 -0.32   0.8    0.14   0.02   0.95     -0.26   0.42   0.71  -0.79  -0.71
  ⋮                       


Вектор b
[-3.88, 3.52, 1.89, 4.18, 0.61, -4.32, 1.93, -4.98, 2.49, 2.39, 1.2, 4.73, 2.52, 0.45, -4.11, -3.8, 3.53, 4.32, -1.82, -0.93, -4.85, -1.81, -2.15, -0.64, 0.11, -4.32, 4.84, -4.61, -4.24, 2.0, 0.33, 3.34, -0.64, -4.87, 1.19, 3.59, -4.04, 0.11, 2.98, 2.99, 4.84, -2.78, -2.29, -3.8, -0.49, -1.96, -0.78, 3.58, 0.09, 2.07, 0.41, 4.84, 4.17, 3.51, -1.14, -1.94, -4.25, 2.63, 2.27, -4.89, 3.37, 0.87, -3.58, 3.76, -2.09, -4.14, 3.16, 4.25, 1.26, 0.31, 3.9, -1.29, 2.82, 0.11, 4.8, -4.3, 0.62, 4.51, 1.54, -4.65, 3.23, -3.59, -1.69, 0.41, 1.68, 3.49, -3.7, -2.43, 1.72, -2.12, -4.08, -1.79, -2.26, 0.32, 4.45, -0.57, -0.32, -3.6, 3.39, -1.73]

Матрица delta_A


100×100 Matrix{Float64}:
  0.01   0.01   0.02  -0.02  -0.01  …  -0.01  -0.01   0.0   -0.01   0.0
  0.01   0.0    0.0   -0.02  -0.02     -0.01  -0.01   0.01   0.0    0.01
 -0.01   0.02   0.0    0.0   -0.02      0.0   -0.02   0.01   0.02   0.02
  0.02  -0.02   0.02   0.01   0.0      -0.02   0.02   0.0    0.02   0.01
 -0.01   0.01   0.01   0.02   0.02     -0.01   0.01  -0.01   0.02   0.01
 -0.01   0.02   0.02  -0.01   0.0   …   0.02   0.01   0.02   0.0    0.0
  0.02  -0.02   0.01   0.0   -0.01      0.02   0.02   0.01   0.01   0.01
  0.0    0.02   0.01  -0.01  -0.02      0.0   -0.02   0.0    0.0   -0.01
  0.01   0.02  -0.02  -0.01   0.02     -0.02   0.01  -0.02   0.0    0.0
 -0.01   0.0    0.0    0.01  -0.02     -0.01   0.01  -0.01   0.02   0.02
 -0.01  -0.02   0.0   -0.02  -0.01  …   0.0    0.01   0.01   0.02  -0.02
  0.02   0.01   0.02   0.0    0.0       0.01   0.01   0.0    0.0    0.02
  0.0    0.0    0.01   0.02   0.02      0.0   -0.01  -0.01  -0.02   0.01
  ⋮                          


Вектор delta_b
[-0.09, 0.01, -0.09, 0.02, -0.09, 0.07, 0.04, 0.06, 0.04, -0.1, -0.07, -0.07, 0.03, 0.1, -0.07, -0.07, 0.0, 0.04, 0.1, -0.06, -0.04, 0.1, 0.06, -0.02, -0.01, -0.02, 0.1, -0.01, -0.08, -0.03, 0.1, 0.03, 0.05, 0.03, 0.04, -0.07, 0.05, 0.04, -0.08, 0.02, 0.09, 0.0, 0.03, -0.01, -0.05, -0.06, 0.08, 0.06, 0.05, 0.04, 0.1, -0.09, -0.07, 0.02, -0.02, 0.1, -0.07, 0.09, -0.02, 0.04, 0.0, 0.01, 0.1, -0.05, -0.1, 0.04, 0.01, 0.03, -0.08, -0.08, -0.02, -0.08, 0.07, 0.06, 0.02, 0.04, 0.05, -0.05, -0.03, -0.08, 0.08, 0.07, -0.05, 0.1, -0.07, 0.08, -0.03, 0.04, -0.07, -0.02, 0.09, -0.01, -0.01, 0.0, -0.04, -0.03, 0.07, 0.1, 0.0, 0.05]

=============== p = 2 ===============
Решение системы x
[1.75, -1.17, 4.04, 4.64, -2.99, 1.91, 1.01, 3.97, 11.28, -3.4, 5.68, 5.92, -1.18, 2.54, 4.33, 0.41, -12.13, 5.83, -1.46, -1.61, 1.26, -3.39, 6.09, -3.27, 1.93, -11.22, -10.17, -1.44, -6.92, -12.49, -0.06, 2.7, 7.7, -5.25, -3.38, 3.95, 3.22, 5.12, -0.43, -3.0, 4.12, 2.35, 1.3, -5.54, -4.21, -4.84, 

In [21]:
# СОЗДАНИЕ ДАННЫХ ДЛЯ ГРАФИКА
function buildSurfaces(A, b, delta_A, delta_b, p)
    x = solveSystem(A, b)
    nu = condNumber(A, p)
    
    # Изменение возмущений от 0 до полного значения
    scale = collect(0.00:0.05:1.00)
    count = length(scale)

    xAxis = zeros(count)
    yAxis = zeros(count)

    directSurf = zeros(count, count)
    estimateSurf = zeros(count, count)
    
    # Значения для оси X
    for j in eachindex(scale)
        cur_b = scale[j] * delta_b
        xAxis[j] = norm(cur_b, p)
    end
    
    # Значения для оси Y
    for i in eachindex(scale)
        cur_A = scale[i] * delta_A
        yAxis[i] = opnorm(cur_A, p)
    end

    # Вычисление значений двух поверхностей
    for i in eachindex(scale)
        cur_A = scale[i] * delta_A
        for j in eachindex(scale)
            cur_b = scale[j] * delta_b

            xNew = solveSystem(A + cur_A,b + cur_b)
            delta_x = xNew - x

            directSurf[i, j] = round((norm(delta_x, p) / norm(x, p)) * 100.00, digits = 2)
            estimateSurf[i, j] = round(nu * (norm(cur_b, p) / norm(b, p) + opnorm(cur_A, p) / opnorm(A, p)) * 100.00, digits = 2)
        end
    end

    return (xAxis, yAxis, directSurf, estimateSurf)
end

# ПОСТРОЕНИЕ ГРАФИКА
function makeGraph(A, b, delta_A, delta_b, p)
    xAxis, yAxis, directSurf, estimateSurf = buildSurfaces(A, b, delta_A, delta_b, p)

    if p == Inf
        pName = "Inf"
    else
        pName = "2"
    end

    graph = surface(
        xAxis, 
        yAxis, 
        directSurf,
        xlabel = "||delta_b||p",
        ylabel = "||delta_A||p",
        zlabel = "delta(x), %",
        title = "p = " * pName,
        label = "Прямой расчет",
        c = :blues,
        seriesalpha = 0.75,
        colorbar = false,
        camera = (35, 25),
        size = (900, 650)
    )

    surface!(
        graph,
        xAxis,
        yAxis,
        estimateSurf,
        label = "Оценка",
        c = :reds,
        seriesalpha = 0.55,
        colorbar = false
    )

    return graph
end

# ГРАФИК ДЛЯ p = 2
graph2 = makeGraph(A, b, delta_A, delta_b, 2)

# ГРАФИК ДЛЯ p = Inf
graphInf = makeGraph(A, b, delta_A, delta_b, Inf)

# ВЫВОД ГРАФИКОВ
display(graph2)
display(graphInf)